In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
movie_language_df = (
    spark.table("databricks_ws.bronze.movie_languages")
    .filter(col("file_date") == file_date)
    )

In [0]:
movie_language_selected_df = (
    movie_language_df.select(
        col("languageId").cast("integer"),
        col("movieId").cast("integer"),
        col("file_date")
        )
    )

In [0]:
movie_with_columns_df = (
    add_ingestion_date(movie_language_selected_df)
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("languageId", "language_id")
    .withColumn("environment", lit("Production"))
)

In [0]:
table_name = "databricks_ws.silver.movie_languages"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            movie_with_columns_df.alias("src"),
            "tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    movie_with_columns_df.write \
        .format("delta") \
        .saveAsTable(table_name)

In [0]:
movie_with_columns_df.display()

language_id,movie_id,ingestion_date,environment
24574,36593,2026-11-07T01:16:09.609353Z,Production
24574,36597,2026-11-07T01:16:09.609353Z,Production
24574,36643,2026-11-07T01:16:09.609353Z,Production
24574,36647,2026-11-07T01:16:09.609353Z,Production
24574,36648,2026-11-07T01:16:09.609353Z,Production
24574,36657,2026-11-07T01:16:09.609353Z,Production
24574,36658,2026-11-07T01:16:09.609353Z,Production
24574,36668,2026-11-07T01:16:09.609353Z,Production
24574,36669,2026-11-07T01:16:09.609353Z,Production
24574,36670,2026-11-07T01:16:09.609353Z,Production
